# **Testing**

In [1]:
! pip install agent-framework --pre


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

In [27]:
Source_Agent = AzureOpenAIChatClient(credential=AzureCliCredential()).as_agent(
    instructions="""
You are a Source Discovery Agent.

Your ONLY responsibility is to identify trustworthy and authoritative
web entry points relevant to a given research topic which will help in gathering links for job preparation.

Rules:
- Do NOT fabricate URLs.
- Prefer official company pages and well-known platforms.
- If unsure, exclude the source.
- You do NOT summarize content.
- You do NOT analyze skills.
- You ONLY return source locations.

Source priority:
1. Official company websites and documentation
2. Official company blogs or engineering portals
3. Official job portals (LinkedIn, Greenhouse, Lever)
4. Trusted learning platforms (MDN, W3Schools, GeeksforGeeks)
5. Reputable industry publications
6. Job portal aggregators (Indeed, Glassdoor, LinkedIn Jobs)
7. Well-known Q&A sites (Stack Overflow)
8. Recognized tech news sites (TechCrunch, The Verge)
9. General search engines (Google, Bing)

Output MUST be valid Python list syntax.

Research topic: "{research_topic}"
""",
    name="SourceDiscoveryAgent"
)

research_topic = "Software Development in IBM"
sources = await Source_Agent.run(f"Identify sources for research on: {research_topic}")
print("Discovered Sources:", sources.text)

Discovered Sources: ```python
[
    "https://www.ibm.com/software",
    "https://www.ibm.com/blogs/",
    "https://www.ibm.com/careers",
    "https://www.linkedin.com/company/ibm/",
    "https://developer.ibm.com/",
    "https://www.ibm.com/cloud/developer",
    "https://www.geeksforgeeks.org/ibm-company/",
    "https://www.indeed.com/q-IBM-jobs.html",
    "https://www.glassdoor.com/Jobs/IBM-Jobs-E354.htm",
    "https://stackoverflow.com/questions/tagged/ibm"
]
```


In [29]:
Research_Agent = AzureOpenAIChatClient(credential=AzureCliCredential()).as_agent(
    instructions="""
    You are a Web Research Extraction Agent.

    Your responsibility is to extract factual, high-level information
    from the provided trusted source locations.

    Rules:
    - Use ONLY the provided sources.
    - Do NOT invent facts.
    - If information is unavailable, state it explicitly.
    - Keep summaries concise and factual.
    - Always cite the source URL for each fact.

    You do NOT:
    - Create study plans
    - Infer skills
    - Make assumptions beyond the text

    Output format:
    - Bullet points
    - Each bullet MUST end with a source reference

    Trusted sources:
    {trusted_sources}
    """,
    name="ResearchExtractionAgent"
    )

info = await Research_Agent.run(f'Research on the topic: {research_topic} from the given trusted sources. {sources.text}')
print("Research Information:", info.text)

Research Information: - IBM offers a range of software solutions, including products and services for industry-specific applications, cloud, AI, and data analytics, among others. [source: https://www.ibm.com/software]
- IBM's developer site provides resources, tools, and documentation to assist developers in building and integrating applications on the IBM platform. [source: https://developer.ibm.com/]
- The IBM Cloud Developer platform offers comprehensive cloud services and tools for developers aiming to create, deploy, and manage applications efficiently in a cloud environment. [source: https://www.ibm.com/cloud/developer]
- Research and insights about IBM's software engineering and development practices can be found on their corporate blog, which discusses technology trends and innovations spearheaded by IBM. [source: https://www.ibm.com/blogs/] 
- IBM careers highlight available job roles, including positions in software development, indicating the company's commitment to attracti

# **Working in 3 layers info extraction**

In [105]:
# --- Prompt Definitions ---

SOURCE_AGENT_PROMPT = """
You are a Source Discovery Agent.
Your goal is to find trustworthy web sources and EXISTING STUDY PLANS to help prepare for a specific job interview.
Target:
- Company: {company_name}
- Job Role: {job_role}

Your responsibility is to identify authoritative web entry points where we can find:
1. Company culture, values, tech stack, and engineering practices.
2. Role-specific expectations, required skills, and day-to-day responsibilities.
3. Interview experiences, question patterns, and specific "Study Guides" or "Roadmaps" created by others for this role.

Search Strategy:
- Prefer official company pages (Careers, Engineering Blogs).
- Prioritize professional networks (LinkedIn, Glassdoor, Blind).
- SEEK OUT community-created study plans/roadmaps on GitHub, Medium, Reddit, or Dev.to.
- Include technical preparation hubs (LeetCode, GeeksforGeeks) specific to [{company_name}].

Output contract (STRICT):
- Return ONLY a valid Python list of URLs strings.
- Example: ["https://careers.google.com", "https://github.com/jdoe/google-interview-roadmap", "https://leetcode.com/company/google"]
"""

In [106]:
# Define Research Target
company_name = "Google"
job_role = "Business Developer"

client = AzureOpenAIChatClient(
    credential=AzureCliCredential()
)

# -------- Agent 1: Source Discovery --------
source_agent = client.as_agent(
    instructions=SOURCE_AGENT_PROMPT.format(company_name=company_name, job_role=job_role),
    name="SourceDiscoveryAgent"
)

sources = await source_agent.run()
print("Discovered Sources:", sources.text)

Discovered Sources: ```python
[
    "https://careers.google.com",
    "https://www.google.com/about/careers/",
    "https://www.glassdoor.com/Overview/Working-at-Google-EI_IE9079.11,17.htm",
    "https://www.linkedin.com/company/google/",
    "https://www.complex.com/life/google-culture-values",
    "https://www.geeksforgeeks.org/google-interview-experience/",
    "https://leetcode.com/company/google",
    "https://medium.com/@zhangjunming/google-interview-roadmap-4ebc82f397b6",
    "https://www.reddit.com/r/cscareerquestions/comments/8lyu4y/google_interview_experience_as_a_business_developer/",
    "https://dev.to/rajeshmanohar/google-interview-preparation-guide-for-business-developer-role-5432"
]
```


In [107]:
TOPIC_AGENT_PROMPT = """
You are a Research & Topic Extraction Agent.
Your Goal: Create a "Key Topics to Prepare" study plan based on the provided trusted sources.

Context:
- Company: {company_name}
- Job Role: {job_role}

Sources to Analyze:
{trusted_sources}

Task:
1. Simulate visiting and reading the provided sources.
2. Extract relevant skills, competencies, and company-specific values.
3. Synthesize this into a structured list of key topics.

Rules:
- Do NOT invent topics not supported by the sources or standard role expectations.
- Differentiate between:
  - "Technical": Hard skills, Tools, Domain Knowledge, Functional Competencies (e.g., Coding, Sales Strategy, Financial Modeling, CRM tools).
  - "Behavioral": Soft skills, Culture fit, Leadership Principles, Communication.

Output contract (STRICT JSON ONLY):
{{
  "technical_topics": [
    {{ "topic": "Name", "importance": "High/Medium", "reason": "Justification from sources" }}
  ],
  "behavioral_topics": [
    {{ "topic": "Name", "importance": "High/Medium", "reason": "Justification from sources" }}
  ]
}}
"""

In [108]:
# -------- Agent 2: Topic Extraction (Direct from Sources) --------
trusted_sources = sources.text

topic_agent = client.as_agent(
    instructions=TOPIC_AGENT_PROMPT.format(
        company_name=company_name, 
        job_role=job_role,
        trusted_sources=trusted_sources
    ),
    name="TopicExtractionAgent"
)

topic_result = await topic_agent.run(
    f"Analyze these sources and generate the key topics to prepare for {job_role} at {company_name}."
)

print("Topics to Cover:", topic_result.text)

Topics to Cover: {
  "technical_topics": [
    {
      "topic": "Sales Strategy",
      "importance": "High",
      "reason": "Business developers are expected to drive sales and implement strategic initiatives in a competitive environment, as noted in various sources discussing the competencies for the role."
    },
    {
      "topic": "Market Analysis",
      "importance": "High",
      "reason": "Understanding market dynamics and being able to assess competitive positioning is crucial for a business developer, which is emphasized in industry discussions and interview experiences."
    },
    {
      "topic": "CRM Tools Proficiency",
      "importance": "Medium",
      "reason": "Proficiency in customer relationship management tools is vital for managing client interactions and sales processes, as mentioned in practical interview guides."
    },
    {
      "topic": "Financial Modeling",
      "importance": "Medium",
      "reason": "A solid grasp of financial modeling is important 

In [109]:
ATOMIC_TOPIC_AGENT_PROMPT = """
You are a Syllabus Decomposition Agent.
Your Goal: Break down high-level study topics into a COMPREHENSIVE list of small, atomic, actionable study units.

Context:
- Company: {company_name}
- Job Role: {job_role}

Input Data:
The user will provide a list of "Technical" (Hard Skills) and "Behavioral" (Soft Skills) topics.

Task:
For each high-level topic, generate an EXHAUSTIVE list of atomic sub-concepts.
- Atomic means: A single concept that can be studied, practiced, or tested in isolation.
- Example (Tech): "System Design" -> ["Load Balancing", "Consistent Hashing", "CAP Theorem"].
- Example (Non-Tech): "Sales Strategy" -> ["Pipeline Management", "Needs Analysis", "Closing Techniques", "Objection Handling"].

Output contract (STRICT JSON ONLY):
{{
  "atomic_study_plan": [
    {{
      "parent_topic": "High Level Topic Name",
      "category": "Technical | Behavioral",
      "atomic_units": [
        "Unit 1",
        "Unit 2",
        "Unit 3"
      ]
    }}
  ]
}}
"""

In [110]:
# -------- Agent 3: Atomic Decomposition --------
topics_json = topic_result.text

atomic_agent = client.as_agent(
    instructions=ATOMIC_TOPIC_AGENT_PROMPT.format(
        company_name=company_name,
        job_role=job_role
    ),
    name="AtomicDecompositionAgent"
)

atomic_result = await atomic_agent.run(
    f"Decompose the following topics into atomic study units:\n{topics_json}"
)

print("Atomic Study Plan:", atomic_result.text)

Atomic Study Plan: {
  "atomic_study_plan": [
    {
      "parent_topic": "Sales Strategy",
      "category": "Behavioral",
      "atomic_units": [
        "Sales Funnel Stages",
        "Pipeline Management Techniques",
        "Needs Analysis",
        "Value Proposition Development",
        "Closing Techniques",
        "Objection Handling",
        "Sales Forecasting Methods",
        "Customer Segmentation Strategies",
        "Sales Performance Metrics"
      ]
    },
    {
      "parent_topic": "Market Analysis",
      "category": "Technical",
      "atomic_units": [
        "Market Segmentation",
        "Competitive Analysis Techniques",
        "SWOT Analysis",
        "PESTEL Analysis",
        "Industry Trend Research",
        "Customer Needs Assessment",
        "Market Size Estimation",
        "Benchmarks and Key Performance Indicators",
        "Data Sources for Market Research"
      ]
    },
    {
      "parent_topic": "CRM Tools Proficiency",
      "category": "Tec

In [111]:
STUDY_PLAN_AGENT_PROMPT = """
You are a Personal Study Scheduler Agent.
Your Goal: Create a detailed week-by-week study schedule using ALL provided atomic study units.

Context:
- Company: {company_name}
- Job Role: {job_role}
- Timeline: {weeks} weeks

Input Data:
The user will provide a comprehensive list of atomic study units (Technical and Behavioral).

Task:
1. Distribute ALL atomic study units logically across {weeks} weeks. DO NOT SKIP ANY TOPICS.
2. Ensure a balanced mix of Technical and Behavioral topics each week.
3. Structure the weeks to progress from Foundations -> Core Concepts -> Advanced -> MockPrep.

Output contract:
- Return a valid Markdown schedule.
- For each week, group related atomic units under their Parent Topic.
- Format:
  ### Week X: [Theme]
  #### [Parent Topic Name]
  - [Atomic Unit 1]
  - [Atomic Unit 2]
  - [Atomic Unit 3]
  ...
  #### [Another Parent Topic]
  ...
"""

In [112]:
# -------- Agent 4: Study Plan Generation --------
atomic_plan_json = atomic_result.text
weeks_for_preparation = 4

plan_agent = client.as_agent(
    instructions=STUDY_PLAN_AGENT_PROMPT.format(
        company_name=company_name,
        job_role=job_role,
        weeks=weeks_for_preparation
    ),
    name="StudyPlanAgent"
)

study_plan_result = await plan_agent.run(
    f"Create a {weeks_for_preparation}-week study plan using these study units:\n{atomic_plan_json}"
)

print("Final Study Plan:", study_plan_result.text)

Final Study Plan: ```markdown
### Week 1: Foundations
#### Sales Strategy
- Sales Funnel Stages
- Pipeline Management Techniques
- Needs Analysis
- Value Proposition Development

#### Market Analysis
- Market Segmentation
- Competitive Analysis Techniques
- SWOT Analysis
- PESTEL Analysis

#### Communication Skills
- Effective Listening Techniques
- Clarity of Expression
- Non-Verbal Communication Skills

#### Adaptability
- Embracing Change Management
- Responding to Unexpected Challenges
- Continuous Learning Mindset

### Week 2: Core Concepts
#### Sales Strategy
- Closing Techniques
- Objection Handling
- Sales Forecasting Methods
- Customer Segmentation Strategies

#### Market Analysis
- Industry Trend Research
- Customer Needs Assessment
- Market Size Estimation
- Benchmarks and Key Performance Indicators

#### CRM Tools Proficiency
- Understanding CRM Software Basics
- Data Entry and Management in CRM
- Creating and Managing Customer Profiles

#### Collaboration and Teamwork
- Bu